# 6. Analise de Sensibilidade (Morris Screen) -- Soja no Parana

Roda o mesmo metodo de Morris do pipeline de milho (`util/SensitivityAnalyzer.py`), mas
usando a variante `SoyMorrisScreeningAnalyzerPR` (`util/SensitivityAnalyzer_soja_pr.py`)
que troca a cultura ativa para `soybean` / `Soybean_VanHeemst_1988` e o calendario para o
da soja no PR, mantendo toda a infraestrutura de amostragem/execucao/analise do WOFOST
reaproveitada sem alteracao.

O resultado (ranking de parametros mais sensiveis por cluster) e salvo tanto em CSV
(para inspecao) quanto em um **JSON de ranking por cluster**, que a etapa 7
(Optimization) carrega dinamicamente -- diferente do pipeline de milho, que tem esse
ranking fixado como dicionario no codigo (`WOFOSTOptimizer.CLUSTER_PARAMS`), aqui ainda
nao existe uma calibracao previa para soja/PR, entao o ranking tem que vir do resultado
desta propria etapa.


In [ ]:
import glob
import json
import os
import sys

import pandas as pd

sys.path.append(os.path.join(os.getcwd(), 'util'))
from SensitivityAnalyzer import NetCDFDataLoader
from SensitivityAnalyzer_soja_pr import SoyMorrisScreeningAnalyzerPR
from utils import WOFOST_bounds
from utils_soja_pr import setup_paths_soja_pr

paths = setup_paths_soja_pr()
DEFAULT_BOUNDS = WOFOST_bounds("all")

N_POINTS_PER_CLUSTER = 30  # PR tem bem menos municipios que o dataset global de milho;
                            # ajuste para cima se sobrar tempo/dados por cluster.

nc_loader = NetCDFDataLoader(paths['COMPLETO'], results_dir=paths['RESULTS'])
analyzer = SoyMorrisScreeningAnalyzerPR(DEFAULT_BOUNDS, paths, nc_loader)


In [ ]:
important_params = analyzer.run_analysis(n_points_per_cluster=N_POINTS_PER_CLUSTER)
important_params


## Consolidar ranking por cluster e salvar para a etapa de Optimization


In [ ]:
path_to_results = os.path.join(paths['RESULTS'], "SA_MORRIS_cluster*_point*.csv")
files = glob.glob(path_to_results)

dfs = [pd.read_csv(f) for f in files]
df_combined = pd.concat(dfs, ignore_index=True).round(4)

df_combined = (
    df_combined.groupby(['cluster_id', 'parameter'])[['mu_star', 'sigma']]
    .mean()
    .reset_index()
    .sort_values(by='mu_star', ascending=False)
)

ranking_por_cluster = {}
for cluster_id in df_combined['cluster_id'].unique():
    df_cluster = (
        df_combined[(df_combined['cluster_id'] == cluster_id) & (df_combined['mu_star'] > 0)]
        .sort_values(by='mu_star', ascending=False)
        .reset_index(drop=True)
    )
    df_cluster['Rank'] = df_cluster.index + 1
    ranking_por_cluster[str(cluster_id)] = df_cluster[['Rank', 'parameter', 'mu_star', 'sigma']].to_dict('records')

ranking_json_path = os.path.join(paths['RESULTS'], 'SA_ranking_by_cluster.json')
with open(ranking_json_path, 'w') as f:
    json.dump(ranking_por_cluster, f, indent=2)

print(f"Ranking por cluster salvo em {ranking_json_path}")
for cluster_id, ranking in ranking_por_cluster.items():
    top5 = [r['parameter'] for r in ranking[:5]]
    print(f"Cluster {cluster_id}: top 5 = {top5}")
